# Independent comparator: the Fed-trained model, zero-shot (Colab)

**Why:** our lexicon and the labelling rules share one parent (the codebook). `gtfintechlab/FOMC-RoBERTa` was trained by *other people* on FOMC text with *their* definitions of hawkish and dovish. If it agrees with your gold labels on Bank of Ghana sentences, that is evidence that does not depend on our codebook.

**Steps**
1. Runtime -> Change runtime type -> **T4 GPU** -> Save.
2. Run the cells in order. When asked, upload `gold_annotation.xlsx` (the notebook reads only the `gold_id` and `sentence` columns, never your labels).
3. The last cell downloads `predictions_fed.csv`. Put it in `data\processed\` on your computer.

The notebook prints label counts only. It never prints your sentences.

Licence: the model is CC BY-NC 4.0 (non-commercial research use). Say so in the README.

In [ ]:
!pip -q install transformers scikit-learn openpyxl

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()          # choose gold_annotation.xlsx
name = next(iter(uploaded))
gold = pd.read_excel(name, sheet_name="labels")[["gold_id", "sentence"]]
print(len(gold), "sentences loaded")

In [ ]:
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)           # should say cuda

MODEL = "gtfintechlab/FOMC-RoBERTa"
tokenizer = AutoTokenizer.from_pretrained(MODEL, do_lower_case=True, do_basic_tokenize=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3).to(device).eval()

# From the model card: LABEL_0 = Dovish, LABEL_1 = Hawkish, LABEL_2 = Neutral
LABELS = {0: "dovish", 1: "hawkish", 2: "neutral"}


@torch.no_grad()
def predict(texts, batch_size=32):
    out = []
    for start in range(0, len(texts), batch_size):
        enc = tokenizer(texts[start : start + batch_size], padding=True, truncation=True,
                        max_length=256, return_tensors="pt").to(device)
        out.append(torch.softmax(model(**enc).logits.float(), dim=-1).cpu().numpy())
    return np.vstack(out)

In [ ]:
# Sanity check on three invented Fed sentences, to confirm the label mapping before trusting anything.
checks = [
    ("The Committee decided to raise the target range for the federal funds rate because inflation remains well above its objective.", "hawkish"),
    ("The Committee decided to lower the target range for the federal funds rate because economic activity has weakened sharply.", "dovish"),
    ("The Committee will hold its next meeting in March at the Board of Governors in Washington.", "neutral"),
]
probs = predict([text for text, _ in checks])
for (text, expected), p in zip(checks, probs):
    got = LABELS[int(p.argmax())]
    print(f"expected {expected:8s} got {got:8s} {p.round(2)}")
assert LABELS[int(probs[0].argmax())] == "hawkish" and LABELS[int(probs[1].argmax())] == "dovish", \
    "The label mapping looks wrong: stop and send this output to your assistant."

In [ ]:
probs = predict(gold["sentence"].tolist())
out = gold[["gold_id"]].copy()
for i, label in LABELS.items():
    out[f"p_{label}"] = probs[:, i].round(5)
out["fed_label"] = [LABELS[i] for i in probs.argmax(axis=1)]
out.to_csv("predictions_fed.csv", index=False)
print(out["fed_label"].value_counts())
files.download("predictions_fed.csv")